# EDA — BH Investment Insights
Exploração inicial das bases antes de escrever o ETL.

**Objetivo:** entender shape, tipos, valores nulos, distribuições e inconsistências de cada fonte.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

RAW = Path('../data/raw')

def quick_profile(path: Path, label: str, nrows: int = 5) -> pd.DataFrame:
    """Carrega CSV e exibe perfil básico."""
    for enc in ('utf-8', 'latin-1', 'cp1252'):
        try:
            df = pd.read_csv(path, dtype=str, encoding=enc, low_memory=False)
            break
        except UnicodeDecodeError:
            continue
    print(f'\n=== {label} ===')
    print(f'Shape: {df.shape}')
    print(f'Colunas: {df.columns.tolist()}')
    print(f'Nulos (%):\n{(df.isnull().mean() * 100).round(1).to_string()}')
    return df

## 1. Atividade Econômica

In [ ]:
df_eco = quick_profile(
    RAW / 'atividade_economica' / 'atividade_economica.csv',
    'Atividade Econômica'
)
df_eco.head()

In [ ]:
# Distribuição de situação das empresas
if 'SITUACAO' in df_eco.columns:
    df_eco['SITUACAO'].value_counts().plot(kind='bar', title='Situação das Empresas')
    plt.tight_layout()
    plt.show()

In [ ]:
# Top 10 bairros por número de empresas
if 'BAIRRO' in df_eco.columns:
    (
        df_eco[df_eco.get('SITUACAO', pd.Series('ATIVA')) == 'ATIVA']
        ['BAIRRO'].value_counts().head(10)
        .plot(kind='barh', title='Top 10 Bairros por Empresas Ativas')
    )
    plt.tight_layout()
    plt.show()

## 2. Pontos de Ônibus

In [ ]:
df_pts = quick_profile(
    RAW / 'pontos_onibus' / 'pontos_onibus.csv',
    'Pontos de Ônibus'
)
df_pts.head()

## 3. Embarques por Ponto

In [ ]:
df_emb = quick_profile(
    RAW / 'embarque_por_ponto' / 'embarque_por_ponto.csv',
    'Embarques por Ponto'
)
df_emb.head()

In [ ]:
# Verifica se CODIGO_PONTO existe em ambos para o join
print('Colunas pontos:', df_pts.columns.tolist())
print('Colunas embarques:', df_emb.columns.tolist())

## 4. Acidentes de Trânsito

In [ ]:
df_acid = quick_profile(
    RAW / 'acidentes_transito' / 'acidentes_transito.csv',
    'Acidentes de Trânsito'
)
df_acid.head()

## 5. Parques e Equipamentos

In [ ]:
df_parq = quick_profile(RAW / 'parques' / 'parques.csv', 'Parques')
df_equip = quick_profile(RAW / 'equipamentos_esportivos' / 'equipamentos_esportivos.csv', 'Equipamentos')
display(df_parq.head())
display(df_equip.head())

## 6. Matriz O-D — Estrutura

In [ ]:
df_od = quick_profile(
    RAW / 'matriz_od' / 'matriz_od.csv',
    'Matriz O-D'
)
print('\nAmostra de H3 IDs:')
print(df_od.iloc[:3, :3].to_string())

In [ ]:
# Verifica esparsidade da matriz
total = df_od.shape[0]
print(f'Total de pares O-D: {total:,}')
print(f'Origens únicas: {df_od.iloc[:, 0].nunique()}')
print(f'Destinos únicos: {df_od.iloc[:, 1].nunique()}')

## 7. Consistência de Nomes de Bairro entre Fontes

In [ ]:
def get_bairros(df: pd.DataFrame) -> set:
    if 'BAIRRO' in df.columns:
        return set(df['BAIRRO'].str.strip().str.upper().dropna().unique())
    return set()

b_eco   = get_bairros(df_eco)
b_pts   = get_bairros(df_pts)
b_acid  = get_bairros(df_acid)
b_parq  = get_bairros(df_parq)
b_equip = get_bairros(df_equip)

print(f'Bairros em Atividade Econômica: {len(b_eco)}')
print(f'Bairros em Pontos de Ônibus:    {len(b_pts)}')
print(f'Bairros em Acidentes:           {len(b_acid)}')
print(f'Bairros em Parques:             {len(b_parq)}')
print(f'Bairros em Equipamentos:        {len(b_equip)}')

if b_eco and b_pts:
    so_eco = b_eco - b_pts
    so_pts = b_pts - b_eco
    print(f'\nBairros só em Econômica (não em Pontos): {len(so_eco)}')
    print(f'Bairros só em Pontos (não em Econômica): {len(so_pts)}')
    print('\nExemplos de divergência:', list(so_eco)[:5])